# Exercise 1 — STFT & mel filterbank, explored

A runnable companion to the lab. Every cell below is the **complete reference solution**, broken into steps with plots so you can see what each piece does. Run top-to-bottom, then go back to `starter.py` and rebuild each function from memory.

> This is the same code path NeMo's `AudioToMelSpectrogramPreprocessor` runs in CUDA — yours is slower but gives the same numbers.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cwkendall/parakeet-study/blob/main/exercises/01-stft-from-scratch/explore.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/cwkendall/parakeet-study/main?labpath=exercises%2F01-stft-from-scratch%2Fexplore.ipynb)

### Setup — install dependencies

In [ ]:
!pip -q install numpy scipy matplotlib librosa

### Imports and helpers

In [ ]:
"""
Exercise 1 — reference solution.

Only consult this when you're stuck on a TODO in starter.py.
"""
from __future__ import annotations

import numpy as np

## The solution, step by step

Each function below is the reference implementation. Read it, run it, then try to reproduce it in `starter.py`.

#### `dft`

In [ ]:
def dft(x: np.ndarray) -> np.ndarray:
    N = len(x)
    k = np.arange(N).reshape(-1, 1)
    n = np.arange(N).reshape(1, -1)
    W = np.exp(-2j * np.pi * k * n / N)
    return W @ x

#### `hann_window`

In [ ]:
def hann_window(n: int) -> np.ndarray:
    if n == 1:
        return np.ones(1)
    return 0.5 * (1 - np.cos(2 * np.pi * np.arange(n) / (n - 1)))

#### `stft`

In [ ]:
def stft(
    x: np.ndarray,
    n_fft: int = 512,
    hop_length: int = 160,
    win_length: int = 400,
) -> np.ndarray:
    assert win_length <= n_fft, "win_length must be <= n_fft (zero-pad otherwise)"
    w = hann_window(win_length)

    if len(x) < win_length:
        n_frames = 1
    else:
        n_frames = 1 + (len(x) - win_length) // hop_length

    n_bins = n_fft // 2 + 1
    out = np.zeros((n_bins, n_frames), dtype=np.complex128)

    pad_left = (n_fft - win_length) // 2
    pad_right = n_fft - win_length - pad_left

    for i in range(n_frames):
        start = i * hop_length
        frame = x[start:start + win_length]
        if len(frame) < win_length:
            frame = np.pad(frame, (0, win_length - len(frame)))
        windowed = frame * w
        padded = np.pad(windowed, (pad_left, pad_right))
        out[:, i] = np.fft.rfft(padded, n=n_fft)

    return out

#### `hz_to_mel`

In [ ]:
def hz_to_mel(hz):
    return 2595.0 * np.log10(1.0 + np.asarray(hz) / 700.0)

#### `mel_to_hz`

In [ ]:
def mel_to_hz(mel):
    return 700.0 * (10.0 ** (np.asarray(mel) / 2595.0) - 1.0)

#### `mel_filterbank`

In [ ]:
def mel_filterbank(
    n_mels: int = 80,
    n_fft: int = 512,
    sample_rate: int = 16000,
    f_min: float = 0.0,
    f_max: float | None = None,
) -> np.ndarray:
    if f_max is None:
        f_max = sample_rate / 2

    mel_points = np.linspace(hz_to_mel(f_min), hz_to_mel(f_max), n_mels + 2)
    hz_points = mel_to_hz(mel_points)
    bin_points = np.floor((n_fft + 1) * hz_points / sample_rate).astype(int)

    fb = np.zeros((n_mels, n_fft // 2 + 1), dtype=np.float64)
    for m in range(1, n_mels + 1):
        f_left = bin_points[m - 1]
        f_centre = bin_points[m]
        f_right = bin_points[m + 1]
        if f_centre > f_left:
            for k in range(f_left, f_centre):
                fb[m - 1, k] = (k - f_left) / (f_centre - f_left)
        if f_right > f_centre:
            for k in range(f_centre, f_right):
                fb[m - 1, k] = (f_right - k) / (f_right - f_centre)
    return fb

#### `log_mel_spectrogram`

In [ ]:
def log_mel_spectrogram(
    x: np.ndarray,
    sample_rate: int = 16000,
    n_fft: int = 512,
    win_length: int = 400,
    hop_length: int = 160,
    n_mels: int = 80,
    log_offset: float = 1e-6,
) -> np.ndarray:
    spec = stft(x, n_fft=n_fft, hop_length=hop_length, win_length=win_length)
    power = np.abs(spec) ** 2
    fb = mel_filterbank(n_mels=n_mels, n_fft=n_fft, sample_rate=sample_rate)
    mel = fb @ power
    return np.log(mel + log_offset)

## Explore — a 440 Hz tone through the whole pipeline

In [ ]:
import numpy as np, matplotlib.pyplot as plt
sr = 16000
t = np.arange(sr) / sr
x = 0.5 * np.sin(2 * np.pi * 440 * t)
log_mel = log_mel_spectrogram(x, sample_rate=sr)
print('log-mel shape:', log_mel.shape)
plt.figure(figsize=(8, 3))
plt.imshow(log_mel, origin='lower', aspect='auto')
plt.xlabel('frame'); plt.ylabel('mel bin'); plt.title('log-mel of 440 Hz tone')
plt.colorbar(); plt.tight_layout(); plt.show()

## Explore — the mel filterbank is just a constant matrix

In [ ]:
fb = mel_filterbank(n_mels=20, n_fft=512, sample_rate=16000)
plt.figure(figsize=(8, 3))
for row in fb:
    plt.plot(row)
plt.xlabel('FFT bin'); plt.ylabel('weight'); plt.title('20 triangular mel filters')
plt.tight_layout(); plt.show()

## Cross-check against librosa

To compare like-for-like we ask librosa for the **HTK** mel scale (the formula our `hz_to_mel` uses), uncentered framing, and unnormalised filters. The two won't be bitwise identical — our reference uses the classic `floor((n_fft+1)·f/sr)` bin mapping while librosa builds exact triangles — but the energy should land in the same place.

In [ ]:
import librosa
ref = librosa.feature.melspectrogram(y=x.astype(np.float32), sr=sr,
        n_fft=512, hop_length=160, win_length=400, n_mels=80, power=2.0,
        center=False, htk=True, norm=None)
ours = np.exp(log_mel)  # back to power
print('shapes  ours/librosa:', ours.shape, ref.shape)
print('peak mel-bin ours:', int(ours.sum(1).argmax()),
      ' librosa:', int(ref.sum(1).argmax()))